
# Работа с данными бизнеса в PySpark

## Шаг 1. Загрузка данных и знакомство с ними

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, 
BooleanType, DoubleType, IntegerType, DateType)

# Создаём Spark-сессию
spark = SparkSession.builder \
    .appName("Create DataFrame with Explicit Schema") \
    .getOrCreate()

# Схема для таблицы audition с учетом реального порядка в CSV
audition_schema = StructType([
    StructField("usage_geo_id", IntegerType(), True),      
    StructField("audition_id", IntegerType(), False),                 
    StructField("puid", StringType(), True),         
    StructField("usage_platform_ru", StringType(), True),    
    StructField("msk_business_dt_str", DateType(), True),  
    StructField("app_version", StringType(), True),         
    StructField("adult_content_flg", BooleanType(), True),   
    StructField("hours", DoubleType(), True),                
    StructField("hours_sessions_long", DoubleType(), True), 
    StructField("kids_content_flg", BooleanType(), True),    
    StructField("main_content_id", StringType(), True),      
    # Незадокументированные столбцы, обнаруженные в данных
    StructField("usage_geo_id_name", StringType(), True),           
    StructField("usage_country_nam", StringType(), True)           
])

# Схема для таблицы content с учетом реального порядка в CSV
content_schema = StructType([
    StructField("main_content_id", StringType(), False),            
    StructField("main_content_type", StringType(), True),           
    StructField("main_content_name", StringType(), True),           
    StructField("main_content_duration_hours", DoubleType(), True), 
    StructField("published_topic_title_list", StringType(), True),  
    
    # Фактически здесь имена авторов, а не их ID. Скорректировал название
    StructField("main_author_name", StringType(), True)                
])

# Считываем данные из csv
audition_df = ( spark.read
		.option("header", False)         
		.schema(audition_schema)   
		.csv("sample_1.csv") )

content_df = ( spark.read
		.option("header", False)        
		.schema(content_schema) 
		.csv("sample_2.csv") )
          
# Выводим строки
audition_df.show(10, vertical=True)
content_df.show(10, vertical=True)

-RECORD 0-----------------------------------
 usage_geo_id        | 162                  
 audition_id         | 0                    
 puid                | 68296628-f9d6-11e... 
 usage_platform_ru   | Станция              
 msk_business_dt_str | 2024-11-26           
 app_version         | null                 
 adult_content_flg   | false                
 hours               | 0.0377777777777777   
 hours_sessions_long | 0.0377777777777777   
 kids_content_flg    | true                 
 main_content_id     | oCURrBKV             
 usage_geo_id_name   | Алматы               
 usage_country_nam   | Казахстан            
-RECORD 1-----------------------------------
 usage_geo_id        | 213                  
 audition_id         | 1                    
 puid                | 682966dc-f9d6-11e... 
 usage_platform_ru   | Станция              
 msk_business_dt_str | 2024-11-26           
 app_version         | null                 
 adult_content_flg   | false                
 hours    

In [2]:
# Выводим схемы

audition_df.printSchema()
content_df.printSchema()

root
 |-- usage_geo_id: integer (nullable = true)
 |-- audition_id: integer (nullable = true)
 |-- puid: string (nullable = true)
 |-- usage_platform_ru: string (nullable = true)
 |-- msk_business_dt_str: date (nullable = true)
 |-- app_version: string (nullable = true)
 |-- adult_content_flg: boolean (nullable = true)
 |-- hours: double (nullable = true)
 |-- hours_sessions_long: double (nullable = true)
 |-- kids_content_flg: boolean (nullable = true)
 |-- main_content_id: string (nullable = true)
 |-- usage_geo_id_name: string (nullable = true)
 |-- usage_country_nam: string (nullable = true)

root
 |-- main_content_id: string (nullable = true)
 |-- main_content_type: string (nullable = true)
 |-- main_content_name: string (nullable = true)
 |-- main_content_duration_hours: double (nullable = true)
 |-- published_topic_title_list: string (nullable = true)
 |-- main_author_name: string (nullable = true)



Описание данных **не вполне соответствует** фактическому содержанию таблицы. 
Столбцы идут **в другом порядке**, и их **содержание** в некоторых случаях **отличается** от описания. Также в первой таблице есть неописанные **дополнительные** столбцы.

Благодаря явному **объявлению схемы** данных во всех столбцах **типы данных корректны** и соответствуют их содержанию.

In [3]:
# Посмотрим кол-во строк:
print(f'audition_df: {audition_df.count()}')
print(f'content_df: {content_df.count()}')

audition_df: 1002896
content_df: 31668


**Объем** строк в таблицах **сильно различается**, что ожидаемо, с учетом того, что в первой таблицы содержатся **логи**. 
Во второй таблице перечислен уникальный контент платформы, что занимает **в разы меньше** строк.

## Шаг 2. Трансформация и преобразование таблиц

In [4]:
# Выбираем нужные столбцы и создаем `minutes_sessions_long`
cut_audition_df = audition_df.select('puid', 'hours_sessions_long', 'msk_business_dt_str', 'adult_content_flg')
cut_audition_df = cut_audition_df.withColumn('minutes_sessions_long', 
                             (F.col('hours_sessions_long') * 60).cast(IntegerType()))

cut_audition_df.select('puid', 'hours_sessions_long', 'minutes_sessions_long').show(10)

+--------------------+-------------------+---------------------+
|                puid|hours_sessions_long|minutes_sessions_long|
+--------------------+-------------------+---------------------+
|68296628-f9d6-11e...| 0.0377777777777777|                    2|
|682966dc-f9d6-11e...|                0.0|                    0|
|682966dc-f9d6-11e...|                0.0|                    0|
|68296704-f9d6-11e...|                0.0|                    0|
|68296722-f9d6-11e...|                0.0|                    0|
|68296740-f9d6-11e...| 0.4890794444444444|                   29|
|68296740-f9d6-11e...|          0.1847275|                   11|
|6829675e-f9d6-11e...| 1.4530555555555555|                   87|
|6829677c-f9d6-11e...| 0.5399818181818182|                   32|
|6829679a-f9d6-11e...|                0.0|                    0|
+--------------------+-------------------+---------------------+
only showing top 10 rows



In [5]:
# добавляем категоризацию по столбцу `is_weekend`
cut_audition_df = cut_audition_df.withColumn('is_weekend',
                    F.when(F.dayofweek(F.col('msk_business_dt_str')) \
                    .isin(1, 7), F.lit(True)) \
                    .otherwise(False))

cut_audition_df.select('puid', 'hours_sessions_long',
                       'minutes_sessions_long', 'is_weekend').show(10)

+--------------------+-------------------+---------------------+----------+
|                puid|hours_sessions_long|minutes_sessions_long|is_weekend|
+--------------------+-------------------+---------------------+----------+
|68296628-f9d6-11e...| 0.0377777777777777|                    2|     false|
|682966dc-f9d6-11e...|                0.0|                    0|     false|
|682966dc-f9d6-11e...|                0.0|                    0|     false|
|68296704-f9d6-11e...|                0.0|                    0|     false|
|68296722-f9d6-11e...|                0.0|                    0|     false|
|68296740-f9d6-11e...| 0.4890794444444444|                   29|     false|
|68296740-f9d6-11e...|          0.1847275|                   11|     false|
|6829675e-f9d6-11e...| 1.4530555555555555|                   87|     false|
|6829677c-f9d6-11e...| 0.5399818181818182|                   32|     false|
|6829679a-f9d6-11e...|                0.0|                    0|     false|
+-----------

In [6]:
# Группируем данные по `is_weekend` и `adult_content_flg`
grouped_audition_df = cut_audition_df.groupBy('is_weekend', 'adult_content_flg').agg(
    F.sum(F.col('minutes_sessions_long')).alias('total_minutes')
    ).dropna().sort('adult_content_flg', 'is_weekend')

grouped_audition_df.show()

+----------+-----------------+-------------+
|is_weekend|adult_content_flg|total_minutes|
+----------+-----------------+-------------+
|     false|            false|      3369978|
|      true|            false|      1401035|
|     false|             true|     14625175|
|      true|             true|      5197958|
+----------+-----------------+-------------+



По минутам активности можно сразу увидеть, что **контент для взрослых превалирует** на плафторме, что говорит о портрете и интересах аудитории. 

Также ожидаемо активность пользователей **поднимается по выходным**, т.к. удельный вес минут просмотра на выходных в буднях в таблице **36%** у взрослого контента и **42%** у детского, а удельный вес количества выходных дней к неделе около **29%**.

Из этих же данных можно сделать вывод, что просмотр **детского контента** на выходных **растет быстрее** взрослого.

## Шаг 3. Соединение таблиц

In [7]:
# Объединяем таблицы
united_df_left = audition_df.join(content_df, on='main_content_id', how='left')
print(united_df_left.count())
united_df_inner = audition_df.join(content_df, on='main_content_id', how='inner')
print(united_df_inner.count())

1002896


996565


In [9]:
# Удаляем лишние столбцы
united_df_left = united_df_left.drop('main_author_name', 'app_version', 'usage_geo_id')
united_df_inner = united_df_inner.drop('main_author_name', 'app_version', 'usage_geo_id')

print(united_df_left.show(10, vertical = True))
print(united_df_inner.show(10, vertical = True))

-RECORD 0-------------------------------------------
 main_content_id             | oCURrBKV             
 audition_id                 | 0                    
 puid                        | 68296628-f9d6-11e... 
 usage_platform_ru           | Станция              
 msk_business_dt_str         | 2024-11-26           
 adult_content_flg           | false                
 hours                       | 0.0377777777777777   
 hours_sessions_long         | 0.0377777777777777   
 kids_content_flg            | true                 
 usage_geo_id_name           | Алматы               
 usage_country_nam           | Казахстан            
 main_content_type           | Audiobook            
 main_content_name           | Фиксики. Интернет... 
 main_content_duration_hours | 0.8302778            
 published_topic_title_list  | 'Детская проза и ... 
-RECORD 1-------------------------------------------
 main_content_id             | qOL0JJL5             
 audition_id                 | 1              

-RECORD 0-------------------------------------------
 main_content_id             | oCURrBKV             
 audition_id                 | 0                    
 puid                        | 68296628-f9d6-11e... 
 usage_platform_ru           | Станция              
 msk_business_dt_str         | 2024-11-26           
 adult_content_flg           | false                
 hours                       | 0.0377777777777777   
 hours_sessions_long         | 0.0377777777777777   
 kids_content_flg            | true                 
 usage_geo_id_name           | Алматы               
 usage_country_nam           | Казахстан            
 main_content_type           | Audiobook            
 main_content_name           | Фиксики. Интернет... 
 main_content_duration_hours | 0.8302778            
 published_topic_title_list  | 'Детская проза и ... 
-RECORD 1-------------------------------------------
 main_content_id             | qOL0JJL5             
 audition_id                 | 1              

In [10]:
# Проверим наличие разницы в кол-ве уникальных пользователей `united_df` и `audition_df` с left join

count_united_left = united_df_left.select('puid').distinct().count()
count_audition = audition_df.select('puid').distinct().count()

if count_united_left  != count_audition:
    print(f'Разница есть: united_df_left - {count_united_left}, audition_df - {count_audition}')
else:
    print('Разницы нет')

Разницы нет


In [11]:
# Проверим наличие разницы в кол-ве уникальных пользователей `united_df` и `audition_df` с inner join

count_united_inner = united_df_inner.select('puid').distinct().count()
count_audition = audition_df.select('puid').distinct().count()

if count_united_inner  != count_audition:
    print(f'Разница есть: count_united_inner - {count_united_inner}, audition_df - {count_audition}')
else:
    print('Разницы нет')

Разница есть: count_united_inner - 31010, audition_df - 31063


При использовании **inner join** у нас появляется разница в количестве уникальных пользователей между `united_df` и `audition_df`. Такая разница, исходя из структуры таблицы, связана **с неполнотой таблицы** `content`. Т.е. в таблице `content` просто нет некоторого вида контента, который есть в таблице `audition`.

При использовании **left join** ожидаемо разницы нет. Такой вид соединения **предпочтительно** использовать для аналитики в данном случае.

In [18]:
# Посмотрим уникальные значения столбца `main_content_type`

unique_rows = united_df_left.select('main_content_type').dropna().distinct().collect()
unique_values = [row[0] for row in unique_rows]
print(f'Уникальные значения `main_content_type`: {unique_values}')

Уникальные значения `main_content_type` ['Audiobook', 'Book', 'Comicbook']
